# Bangla Fake News Detection + Explainability (Kaggle)

**Before Run All:** GPU **T4** · Internet **On**. Both datasets are pulled automatically with `kagglehub`
(`cryptexcode/banfakenews` + `hrithikmajumdar/bangla-fake-news`); you can also attach them via *Add Input*.
Optional: Add-ons → Google Cloud (Cloud Storage) for checkpoint backup/resume, and secret `HF_TOKEN` to publish.

Pipeline (~1–1.5 h): merge + dedupe both datasets → TF-IDF baseline → BanglaBERT with focal loss (checkpoint each
epoch → bucket) → pooled test + BanFakeNews-2.0 independent test → SHAP → `.pth`, HF model, Streamlit Space, report.

## 1 · Setup

In [ ]:
!pip install -q shap py7zr "huggingface_hub>=1.3,<2.0" 2>&1 | tail -1
import importlib.metadata as md, torch
print({p: md.version(p) for p in ("transformers", "huggingface_hub", "shap", "torch")}, "| GPUs:", torch.cuda.device_count())

## 2 · Project code
These cells write the package to `/kaggle/working/repo`. No need to edit them; the steps further down call this code.

In [ ]:
!mkdir -p /kaggle/working/repo/src /kaggle/working/repo/app /kaggle/working/repo/space

In [ ]:
%%writefile /kaggle/working/repo/src/__init__.py
"""package"""


In [ ]:
%%writefile /kaggle/working/repo/src/config.py
from __future__ import annotations

from dataclasses import asdict, dataclass, field


@dataclass
class Config:
    # --- data: both datasets are used together (deduplicated) ---
    kaggle_datasets: list = field(default_factory=lambda: ["cryptexcode/banfakenews",          # BanFakeNews (LREC 2020)
                                                           "hrithikmajumdar/bangla-fake-news"])  # BanFakeNews-2.0
    independent_test_url: str = "https://raw.githubusercontent.com/Shibu4064/IndoNLP/main/New_Test_Dataset.csv"
    local_dirs: list = field(default_factory=lambda: ["/kaggle/input"])  # also scanned (datasets added via "Add Input")
    use_github_v2: bool = True            # Kaggle copy of BanFakeNews-2.0 incomplete? -> authors' GitHub .7z release
    v2_min_rows: int = 20000
    v2_github_base: str = "https://raw.githubusercontent.com/Shibu4064/IndoNLP/main/Datasets/"
    v2_github_files: tuple = ("train_cleaned.7z", "val_cleaned.7z", "test_cleaned.7z")
    max_chars: int = 3000                 # headline + first part of the article
    split: tuple = (0.8, 0.1, 0.1)
    seed: int = 42

    # --- model ---
    model_name: str = "csebuetnlp/banglabert"
    max_len: int = 384
    epochs: int = 3
    lr: float = 3e-5
    batch_size: int = 16
    grad_accum: int = 1
    loss: str = "focal"                   # focal | weighted_ce | ce   (imbalance handling)
    focal_gamma: float = 2.0
    focal_alpha_fake: float = 0.75        # weight of the minority (fake) class in focal loss

    # --- baseline ---
    run_baseline: bool = True             # TF-IDF + logistic regression

    # --- explainability ---
    shap_n: int = 40                      # test articles explained with SHAP
    shap_words: int = 120                 # words per article fed to SHAP (cost grows with length)
    shap_max_evals: int = 300

    # --- outputs ---
    work_dir: str = "/kaggle/working/bfn"
    hub_model_repo: str = ""              # e.g. "your-hf-user/banglabert-fake-news"  (needs HF_TOKEN secret)
    hub_space_repo: str = ""              # e.g. "your-hf-user/bangla-fake-news-demo" (Streamlit, Docker Space)
    hf_token_secret: str = "HF_TOKEN"

    # --- checkpoints in your GCS bucket (same as the other projects) -> resume after a crash ---
    project_id: str = ""
    bucket: str = ""
    bucket_location: str = "US"
    gcs_prefix: str = "bangla-fake-news"
    run_name: str = "main"
    local_store: str = ""

    def to_dict(self):
        return asdict(self)


In [ ]:
%%writefile /kaggle/working/repo/src/data.py
"""Load BanFakeNews + BanFakeNews-2.0, harmonise columns, deduplicate across both, split.
Internally label 1 = FAKE (the class we want to detect); the source files use 1 = authentic."""
from __future__ import annotations

import hashlib
import logging
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

log = logging.getLogger(__name__)
HEAD = ("headline", "title", "heading")
BODY = ("content", "text", "article", "news", "body", "description")
LABEL = ("label", "class", "target", "is_fake")
CAT = ("category", "topic")


def _col(cols, names):
    low = {c.lower().strip(): c for c in cols}
    for n in names:
        if n in low:
            return low[n]
    for n in names:
        for k, c in low.items():
            if n in k:
                return c
    return None


def normalize(t) -> str:
    t = unicodedata.normalize("NFC", str(t) if t is not None and t == t else "")
    t = re.sub(r"https?://\S+|www\.\S+", " ", t)
    return re.sub(r"\s+", " ", t).strip()


def _label(v):
    """Source convention: 1 / 'authentic' / 'real' / 'true' = authentic, 0 / 'fake' = fake. Returns is_fake."""
    s = str(v).strip().lower()
    if s in ("1", "1.0", "authentic", "real", "true"):
        return 0
    if s in ("0", "0.0", "fake", "false"):
        return 1
    return None


def read_csv_any(path: Path) -> pd.DataFrame | None:
    """News text often contains newlines/quotes that break the fast C parser and silently drop rows.
    Try several parsers and keep the one that yields the most rows with a valid 0/1 label."""
    import csv
    import sys
    csv.field_size_limit(min(sys.maxsize, 2 ** 31 - 1))
    tries = [dict(), dict(engine="python"), dict(engine="python", escapechar="\\"),
             dict(engine="python", quoting=csv.QUOTE_NONE, sep=None)]
    best, best_n = None, -1
    for enc in ("utf-8", "utf-8-sig"):
        for kw in tries:
            try:
                df = pd.read_csv(path, encoding=enc, on_bad_lines="skip", **({"low_memory": False} if not kw else {}), **kw)
            except Exception:
                continue
            lc = _col(df.columns, LABEL)
            n = int(fake_labels(df[lc])[0].notna().sum()) if lc else 0
            if n > best_n:
                best, best_n = df, n
        if best_n > 0:
            break
    if best is None:
        log.warning("skip %s: unreadable", path)
        return None
    mb = path.stat().st_size / 1e6
    log.info("    parsed %s: %d labelled rows (%.1f MB, %.1f KB/row)", path.name, best_n, mb, 1000 * mb / max(best_n, 1))
    return best


def fake_labels(series: pd.Series) -> tuple[pd.Series, pd.Series]:
    """-> (is_fake, subtype). BanFakeNews(v1): 1 = authentic, 0 = fake.
    BanFakeNews-2.0 *_cleaned files use 4 classes: 3 = authentic (~80%, regular news), 0/1/2 = fake sub-types
    (satire-like / fabricated / clickbait-misleading by inspection). Mapping inferred from class sizes + examples."""
    v = pd.to_numeric(series, errors="coerce")
    vals = set(v.dropna().astype(int).unique())
    if vals and vals <= {0, 1, 2, 3} and vals & {2, 3}:
        is_fake = v.map(lambda x: None if pd.isna(x) else int(int(x) != 3))
        sub = v.map(lambda x: "authentic" if x == 3 else f"v2_type{int(x)}" if not pd.isna(x) else None)
        return is_fake, sub
    is_fake = series.map(_label)
    return is_fake, is_fake.map(lambda x: None if x is None or pd.isna(x) else ("fake" if x else "authentic"))


def harmonise(df: pd.DataFrame, origin: str, file: str) -> pd.DataFrame | None:
    h, b, l, c = (_col(df.columns, x) for x in (HEAD, BODY, LABEL, CAT))
    if l is None or (h is None and b is None):
        return None
    is_fake, sub = fake_labels(df[l])
    out = pd.DataFrame({
        "headline": df[h].map(normalize) if h else "",
        "content": df[b].map(normalize) if b else "",
        "label": is_fake,
        "subtype": sub,
        "category": df[c].astype(str) if c else "unknown",
    })
    out["origin"], out["file"] = origin, file
    out = out.dropna(subset=["label"])
    out["label"] = out["label"].astype(int)
    return out[(out.headline.str.len() + out.content.str.len()) > 20]


def find_csvs(root: Path) -> list[Path]:
    return sorted(p for p in Path(root).rglob("*.csv") if p.stat().st_size > 0)


def download(cfg) -> dict[str, Path]:
    roots = {}
    try:
        import kagglehub
        for ds in cfg.kaggle_datasets:
            roots[ds] = Path(kagglehub.dataset_download(ds))
            log.info("kagglehub %s -> %s", ds, roots[ds])
    except Exception as e:
        log.warning("kagglehub unavailable (%s); scanning %s", e, cfg.local_dirs)
    for d in cfg.local_dirs:
        if Path(d).exists():
            roots.setdefault(f"local:{d}", Path(d))
    return roots


def origin_of(path: Path) -> str:
    n = str(path).lower()
    return "v2" if ("hrithik" in n or "bangla-fake-news" in n or "banfakenews-2" in n or "_cleaned" in n) else "v1"


def github_v2(cfg) -> Path | None:
    """Official BanFakeNews-2.0 release (train/val/test_cleaned.7z) from the authors' GitHub."""
    try:
        import py7zr
        import requests
    except ImportError:
        log.warning("py7zr not installed -> skipping the GitHub copy of BanFakeNews-2.0")
        return None
    out = Path(cfg.work_dir) / "data" / "banfakenews-2.0-github"
    out.mkdir(parents=True, exist_ok=True)
    for name in cfg.v2_github_files:
        if list(out.glob(name.replace(".7z", "*.csv"))):
            continue
        r = requests.get(cfg.v2_github_base + name, timeout=300)
        r.raise_for_status()
        arc = out / name
        arc.write_bytes(r.content)
        with py7zr.SevenZipFile(arc, "r") as z:
            z.extractall(out)
        arc.unlink()
        log.info("GitHub BanFakeNews-2.0: %s extracted", name)
    return out


def load_all(cfg, roots: dict[str, Path]) -> tuple[pd.DataFrame, pd.DataFrame | None]:
    parts, indep, seen = [], None, set()

    def scan(root):
        nonlocal indep
        for p in find_csvs(root):
            sig = (p.name, p.stat().st_size)
            if sig in seen:                       # same file mounted under another path
                continue
            seen.add(sig)
            raw = read_csv_any(p)
            h = harmonise(raw, origin_of(p), p.name) if raw is not None else None
            if h is None or h.empty:
                log.info("  %s: no usable text/label rows, skipped", p.name)
                continue
            if "new_test" in p.name.lower():
                indep = h
                continue
            log.info("  %s [%s]: %d rows (fake %.1f%%)", p.name, h.origin.iloc[0], len(h), 100 * h.label.mean())
            parts.append(h)
    for _, root in roots.items():
        scan(root)
    n_v2 = sum(len(x) for x in parts if x.origin.iloc[0] == "v2")
    if cfg.use_github_v2 and n_v2 < cfg.v2_min_rows:
        log.warning("Only %d BanFakeNews-2.0 rows found on Kaggle (paper: ~60k) -> trying the authors' GitHub release", n_v2)
        gh = github_v2(cfg)
        if gh is not None:
            parts = [x for x in parts if x.origin.iloc[0] != "v2"]          # replace the incomplete copy
            seen.clear()
            scan(gh)
    if indep is None and cfg.independent_test_url:
        try:
            import io

            import requests
            r = requests.get(cfg.independent_test_url, timeout=60)
            r.raise_for_status()
            indep = harmonise(pd.read_csv(io.BytesIO(r.content)), "v2", "New_Test_Dataset.csv")
            log.info("independent test set: %d rows (fake %.1f%%)", len(indep), 100 * indep.label.mean())
        except Exception as e:
            log.warning("independent test set not available: %s", e)
    if not parts:
        raise RuntimeError("No usable CSVs found. Add the two Kaggle datasets as inputs or check internet access.")
    return pd.concat(parts, ignore_index=True), indep


def _key(h, c):
    s = re.sub(r"\s+", "", (h + c)[:2000])
    return hashlib.md5(s.encode("utf-8")).hexdigest()


def dedupe(df: pd.DataFrame, indep: pd.DataFrame | None, cfg) -> tuple[pd.DataFrame, pd.DataFrame | None]:
    df = df.copy()
    df["key"] = [_key(h, c) for h, c in zip(df.headline, df.content)]
    agg = df.groupby("key").agg(label_n=("label", "nunique"), label=("label", "first"), headline=("headline", "first"),
                                content=("content", "first"), category=("category", "first"), subtype=("subtype", "first"),
                                origin=("origin", lambda s: "+".join(sorted(set(s)))))
    n_conf = int((agg.label_n > 1).sum())
    agg = agg[agg.label_n == 1].drop(columns="label_n").reset_index()
    log.info("Dedup: %d rows -> %d unique articles (%d with conflicting labels dropped)", len(df), len(agg), n_conf)
    if indep is not None:
        indep = indep.copy()
        indep["key"] = [_key(h, c) for h, c in zip(indep.headline, indep.content)]
        overlap = agg.key.isin(set(indep.key))
        log.info("Removed %d training-pool articles that also appear in the independent test set", int(overlap.sum()))
        agg = agg[~overlap]
    return agg, indep


def text_of(df: pd.DataFrame, max_chars: int) -> pd.Series:
    return (df.headline.fillna("") + " । " + df.content.fillna("")).str.slice(0, max_chars)


def split(df: pd.DataFrame, cfg) -> pd.DataFrame:
    """Stratified by (label, origin) so both datasets and both classes appear in every split."""
    rng = np.random.default_rng(cfg.seed)
    df = df.copy()
    df["split"] = "train"
    tr, va, _ = cfg.split
    for _, g in df.groupby(["label", "origin"]):
        idx = rng.permutation(g.index.to_numpy())
        n = len(idx)
        df.loc[idx[int(tr * n):int((tr + va) * n)], "split"] = "val"
        df.loc[idx[int((tr + va) * n):], "split"] = "test"
    return df


def build(cfg, roots=None) -> tuple[pd.DataFrame, pd.DataFrame | None]:
    cache = Path(cfg.work_dir) / "data"
    if (cache / "pool_v4.parquet").exists():
        pool = pd.read_parquet(cache / "pool_v4.parquet")
        ind = pd.read_parquet(cache / "indep_v4.parquet") if (cache / "indep_v4.parquet").exists() else None
        return pool, ind
    raw, indep = load_all(cfg, roots if roots is not None else download(cfg))
    pool, indep = dedupe(raw, indep, cfg)
    pool = split(pool, cfg)
    pool["text"] = text_of(pool, cfg.max_chars)
    if indep is not None:
        indep["text"] = text_of(indep, cfg.max_chars)
    cache.mkdir(parents=True, exist_ok=True)
    pool.to_parquet(cache / "pool_v4.parquet", index=False)
    if indep is not None:
        indep.to_parquet(cache / "indep_v4.parquet", index=False)
    log.info("Pool %d | fake %.1f%% | %s", len(pool), 100 * pool.label.mean(),
             pool.groupby(["split", "origin"]).label.agg(["count", "mean"]).round(3).to_dict())
    return pool, indep


def fingerprint(pool: pd.DataFrame) -> str:
    """Short hash of the training data -> checkpoints from a different dataset are never resumed."""
    h = hashlib.md5(("".join(sorted(pool.key)) + str(pool.label.sum())).encode()).hexdigest()
    return h[:8]


In [ ]:
%%writefile /kaggle/working/repo/src/evaluate.py
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.metrics import (average_precision_score, f1_score, precision_score, recall_score, roc_auc_score,
                             roc_curve)


def best_threshold(y, p) -> float:
    ts = np.linspace(0.05, 0.95, 91)
    return float(ts[int(np.argmax([f1_score(y, p >= t, zero_division=0) for t in ts]))])


def metrics(y, p, thr) -> dict:
    y, p = np.asarray(y), np.asarray(p)
    pred = p >= thr
    out = dict(n=len(y), n_fake=int(y.sum()), fake_f1=f1_score(y, pred, zero_division=0),
               macro_f1=f1_score(y, pred, average="macro", zero_division=0),
               fake_precision=precision_score(y, pred, zero_division=0), fake_recall=recall_score(y, pred, zero_division=0))
    if 0 < y.sum() < len(y):
        out.update(auroc=roc_auc_score(y, p), auprc=average_precision_score(y, p))
    return out


def table(name, sets: dict, thr) -> pd.DataFrame:
    return pd.DataFrame([dict(model=name, eval_set=k, **metrics(d.label, d.p, thr)) for k, d in sets.items()
                         if len(d) and d.label.nunique() > 1])


def by_group(test: pd.DataFrame, col, thr, min_n=50) -> pd.DataFrame:
    rows = []
    for g, d in test.groupby(col):
        if len(d) >= min_n and d.label.nunique() > 1:
            rows.append(dict(**{col: g}, **metrics(d.label, d.p, thr)))
    return pd.DataFrame(rows)


def plot_pr_roc(curves: dict, path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from sklearn.metrics import precision_recall_curve
    fig, ax = plt.subplots(1, 2, figsize=(9, 4))
    for name, (y, p) in curves.items():
        fpr, tpr, _ = roc_curve(y, p)
        pr, rc, _ = precision_recall_curve(y, p)
        ax[0].plot(fpr, tpr, label=f"{name} ({roc_auc_score(y, p):.3f})")
        ax[1].plot(rc, pr, label=f"{name} ({average_precision_score(y, p):.3f})")
    ax[0].plot([0, 1], [0, 1], "k--", lw=.7); ax[0].set_xlabel("FPR"); ax[0].set_ylabel("TPR"); ax[0].set_title("ROC (AUROC)")
    ax[1].set_xlabel("Recall (fake)"); ax[1].set_ylabel("Precision (fake)"); ax[1].set_title("PR (AUPRC)")
    for a in ax:
        a.legend(fontsize=7)
    fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)


def detection_by_subtype(test: pd.DataFrame, thr, min_n=10) -> pd.DataFrame:
    """Share of each fake sub-type flagged as FAKE (recall) and of authentic articles wrongly flagged."""
    if "subtype" not in test:
        return pd.DataFrame()
    rows = []
    for g, d in test.groupby(test.subtype.fillna("unknown")):
        if len(d) >= min_n:
            rate = float((d.p >= thr).mean())
            rows.append(dict(subtype=g, n=len(d), flagged_as_fake=rate))
    return pd.DataFrame(rows).sort_values("subtype")


In [ ]:
%%writefile /kaggle/working/repo/src/explain.py
"""SHAP explanations: which words push an article towards FAKE."""
from __future__ import annotations

import logging
from collections import defaultdict

import numpy as np
import pandas as pd

from .train import predict

log = logging.getLogger(__name__)


def explainer(model, tok, max_len):
    import shap

    def f(texts):
        p = predict(model, tok, [str(t) for t in texts], max_len)
        return np.stack([1 - p, p], 1)
    return shap.Explainer(f, shap.maskers.Text(tok), output_names=["AUTHENTIC", "FAKE"])


def shorten(t: str, n_words: int) -> str:
    return " ".join(str(t).split()[:n_words])


def explain(model, tok, texts: list[str], cfg):
    ex = explainer(model, tok, cfg.max_len)
    return ex([shorten(t, cfg.shap_words) for t in texts], max_evals=cfg.shap_max_evals, batch_size=32)


def html_report(sv, meta: pd.DataFrame) -> str:
    import shap
    blocks = []
    for i in range(len(meta)):
        r = meta.iloc[i]
        blocks.append(f"<h3>#{i + 1} — true: {'FAKE' if r.label else 'AUTHENTIC'} · predicted P(fake) = {r.p:.2f}</h3>"
                      + shap.plots.text(sv[i, :, 1], display=False))
    return ("<html><head><meta charset='utf-8'><title>SHAP — Bangla fake news</title></head><body>"
            "<p>Red words push the prediction towards FAKE, blue towards AUTHENTIC.</p>" + "<hr>".join(blocks) + "</body></html>")


def global_tokens(sv, min_count=3, top=25) -> pd.DataFrame:
    agg = defaultdict(list)
    for i in range(len(sv)):
        for tok, v in zip(sv[i].data, sv[i].values[:, 1]):
            t = str(tok).strip()
            if len(t) >= 2 and not t.isdigit():
                agg[t].append(float(v))
    rows = [(t, np.mean(v), len(v)) for t, v in agg.items() if len(v) >= min_count]
    df = pd.DataFrame(rows, columns=["token", "mean_shap_to_fake", "count"])
    return pd.concat([df.nlargest(top, "mean_shap_to_fake"), df.nsmallest(top, "mean_shap_to_fake")])


FONT_URL = "https://github.com/google/fonts/raw/main/ofl/notosansbengali/NotoSansBengali%5Bwdth,wght%5D.ttf"


def _bangla_font():
    """Best effort: register Noto Sans Bengali so token labels are not empty boxes."""
    from pathlib import Path

    from matplotlib import font_manager
    p = Path("/tmp/NotoSansBengali.ttf")
    try:
        if not p.exists():
            import requests
            r = requests.get(FONT_URL, timeout=30)
            r.raise_for_status()
            p.write_bytes(r.content)
        font_manager.fontManager.addfont(str(p))
        return font_manager.FontProperties(fname=str(p)).get_name()
    except Exception:
        return None


def plot_tokens(g: pd.DataFrame, path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fam = _bangla_font()
    g = g.sort_values("mean_shap_to_fake")
    fig, ax = plt.subplots(figsize=(7, 0.22 * len(g) + 1))
    ax.barh(g.token, g.mean_shap_to_fake, color=["#c0392b" if v > 0 else "#2e86c1" for v in g.mean_shap_to_fake])
    ax.set_xlabel("mean SHAP towards FAKE")
    ax.set_title("Words that push towards FAKE (red) / AUTHENTIC (blue)")
    if fam:
        for lbl in ax.get_yticklabels():
            lbl.set_fontfamily(fam)
    fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)


In [ ]:
%%writefile /kaggle/working/repo/src/gcp.py
"""BigQuery helpers. Works with Kaggle's "Google Cloud Services" add-on
(linked account) and with normal ADC credentials elsewhere."""
from __future__ import annotations

import json
import logging
import os
from pathlib import Path

log = logging.getLogger(__name__)
SOURCE_PROJECT = "physionet-data"


def _kaggle_integrations() -> str:
    return os.environ.get("KAGGLE_KERNEL_INTEGRATIONS", "")


def credentials(target: str):
    """Credentials for 'bigquery' or 'gcs'.
    On Kaggle: the account attached in Add-ons -> Google Cloud Services (passed explicitly, so we never
    fall back to the GCE metadata server). Elsewhere: Application Default Credentials (None)."""
    if "KAGGLE_USER_SECRETS_TOKEN" not in os.environ:
        return None                                   # not on Kaggle -> ADC
    want = {"bigquery": "BIGQUERY", "gcs": "GCS"}[target]
    have = [x.upper() for x in _kaggle_integrations().split(":") if x]
    if want in have:
        from kaggle_secrets import GcpTarget
        try:
            from kaggle_gcp import KaggleKernelCredentials
        except ImportError:                           # minimal re-implementation
            from google.auth import credentials as gac
            from kaggle_secrets import UserSecretsClient

            class KaggleKernelCredentials(gac.Credentials):
                def __init__(self, target):
                    super().__init__()
                    self.target = target

                def refresh(self, request):
                    self.token, self.expiry = UserSecretsClient()._get_access_token(self.target)
        return KaggleKernelCredentials(target=GcpTarget[want])
    try:                                              # Add-ons -> "Google Cloud SDK" also works
        from google.oauth2.credentials import Credentials
        from kaggle_secrets import UserSecretsClient
        info = json.loads(UserSecretsClient().get_gcloud_credential())
        log.info("Using Google Cloud SDK add-on credential for %s", target)
        return Credentials.from_authorized_user_info(info)
    except Exception:
        pass
    raise RuntimeError(
        f"Kaggle has no {want} integration attached (KAGGLE_KERNEL_INTEGRATIONS='{_kaggle_integrations()}').\n"
        "Fix: Add-ons -> Google Cloud Services -> attach your Google account and tick BOTH 'BigQuery' and "
        "'Cloud Storage' -> Save -> then Run -> Restart & clear outputs -> Run All.")


def validate_project_id(project_id: str) -> str:
    pid = (project_id or "").strip()
    if not pid or "YOUR_" in pid.upper() or " " in pid:
        raise ValueError(
            f"cfg.project_id='{project_id}' is not a real GCP project ID.\n"
            "Find it: console.cloud.google.com -> project picker (top bar) -> copy the 'ID' column "
            "(e.g. 'mimic-research-123456', NOT the display name). Use the same Google account/project "
            "that is attached in Kaggle Add-ons and has billing/BigQuery enabled.")
    return pid


def check_access(project_id: str, probes=()):
    """Quick pre-flight check printed in the notebook."""
    try:
        validate_project_id(project_id)
        print("project_id:", project_id, "OK")
    except ValueError as e:
        print("project_id: FAILED ->", e)
        return
    print("Kaggle integrations:", _kaggle_integrations() or "(none)")
    for t in ("bigquery", "gcs"):
        try:
            c = credentials(t)
            if c is not None:
                from google.auth.transport.requests import Request
                c.refresh(Request())
            print(f"  {t:8s}: OK")
        except Exception as e:
            print(f"  {t:8s}: FAILED -> {e}")
            return
    try:
        client = bq_client(project_id)
        from google.cloud import bigquery
        client.query("SELECT 1", job_config=bigquery.QueryJobConfig(dry_run=True))
        print("  BigQuery project/billing: OK")
    except Exception as e:
        print("  BigQuery project/billing: FAILED ->", str(e)[:300])
        return
    for t in probes:
        ok = _dry_run_ok(client, f"SELECT 1 FROM `{SOURCE_PROJECT}.{t}` LIMIT 1")
        print(f"  PhysioNet {t}: {'OK' if ok else 'not readable (other version name is tried automatically)'}")


def bq_client(project_id: str):
    from google.cloud import bigquery
    project_id = validate_project_id(project_id)
    return bigquery.Client(project=project_id, credentials=credentials("bigquery"))


def _dry_run_ok(client, sql: str) -> bool:
    from google.api_core import exceptions as gexc
    from google.cloud import bigquery
    try:
        client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
        return True
    except (gexc.NotFound, gexc.Forbidden, gexc.BadRequest):
        return False            # dataset absent or no PhysioNet access -> try next candidate


def resolve_dataset(client, override: str, candidates: list[str], probe_table: str) -> str:
    """Return 'physionet-data.<dataset>' that the account can read (free dry-run probe)."""
    names = [override] if override else candidates
    for name in names:
        fq = name if "." in name else f"{SOURCE_PROJECT}.{name}"
        if _dry_run_ok(client, f"SELECT 1 FROM `{fq}.{probe_table}` LIMIT 1"):
            log.info("Using dataset %s", fq)
            return fq
    raise RuntimeError(
        f"None of {names} is readable (probe table '{probe_table}'). Open BigQuery console -> "
        f"'{SOURCE_PROJECT}' and set the dataset name in the config.")


def query_df(client, sql: str):
    job = client.query(sql)
    job.result()
    log.info("BigQuery processed %.2f GB", (job.total_bytes_processed or 0) / 1e9)
    try:
        return job.to_dataframe(create_bqstorage_client=True)
    except Exception as e:  # Storage Read API not permitted -> REST fallback
        log.warning("BQ Storage API unavailable (%s). Falling back to REST download.", type(e).__name__)
        return job.result().to_dataframe(create_bqstorage_client=False)


In [ ]:
%%writefile /kaggle/working/repo/src/hub.py
from __future__ import annotations

from pathlib import Path

REPO = Path(__file__).resolve().parents[1]


def push_all(cfg, out: Path, token: str) -> dict:
    from huggingface_hub import HfApi
    api, res = HfApi(token=token), {}
    if cfg.hub_model_repo:
        api.create_repo(cfg.hub_model_repo, exist_ok=True)
        (out / "model" / "hf" / "README.md").write_text((REPO / "space" / "MODEL_CARD.md").read_text(encoding="utf-8"),
                                                        encoding="utf-8")
        api.upload_folder(folder_path=str(out / "model" / "hf"), repo_id=cfg.hub_model_repo)
        api.upload_file(path_or_fileobj=str(out / "model" / "banglabert_fake_news.pth"),
                        path_in_repo="banglabert_fake_news.pth", repo_id=cfg.hub_model_repo)
        res["model"] = f"https://huggingface.co/{cfg.hub_model_repo}"
    if cfg.hub_space_repo:
        api.create_repo(cfg.hub_space_repo, repo_type="space", space_sdk="docker", exist_ok=True)
        for f in ("Dockerfile", "README.md", "requirements.txt"):
            api.upload_file(path_or_fileobj=str(REPO / "space" / f), path_in_repo=f, repo_id=cfg.hub_space_repo,
                            repo_type="space")
        api.upload_file(path_or_fileobj=str(REPO / "app" / "streamlit_app.py"), path_in_repo="streamlit_app.py",
                        repo_id=cfg.hub_space_repo, repo_type="space")
        if cfg.hub_model_repo:
            api.add_space_variable(cfg.hub_space_repo, "MODEL_ID", cfg.hub_model_repo)
        res["space"] = f"https://huggingface.co/spaces/{cfg.hub_space_repo}"
    return res


In [ ]:
%%writefile /kaggle/working/repo/src/pipeline.py
"""data (both datasets) -> TF-IDF baseline -> BanglaBERT (focal loss) -> eval (pooled test + independent test)
-> SHAP explanations -> .pth / HF export -> report."""
from __future__ import annotations

import json
import logging
import os
import time
from pathlib import Path

import pandas as pd
import torch

from . import data, evaluate, explain
from .config import Config
from .train import baseline, history, predict, train

log = logging.getLogger("bfn")


def _hf_token(cfg):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(cfg.hf_token_secret)
    except Exception:
        return os.environ.get(cfg.hf_token_secret)


def run(cfg: Config, roots=None) -> dict:
    t0 = time.time()
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s", force=True)
    for noisy in ("httpx", "httpcore", "huggingface_hub", "urllib3", "filelock"):
        logging.getLogger(noisy).setLevel(logging.WARNING)
    out = Path(cfg.work_dir)
    for d in ("figures", "model", "report", "tables"):
        (out / d).mkdir(parents=True, exist_ok=True)
    store = None
    if cfg.project_id or cfg.local_store:
        from .store import make_store
        store = make_store(cfg)

    # %% Load + merge both datasets | BanFakeNews + BanFakeNews-2.0 → dedupe → stratified split. The independent test set stays separate.
    # show: pool.groupby(["split", "origin"]).label.agg(["count", "mean"]).rename(columns={"mean": "fake_rate"})
    pool, indep = data.build(cfg, roots)
    tr, va, te = (pool[pool.split == s] for s in ("train", "val", "test"))
    sets = {"pooled test": te}
    if indep is not None and len(indep):
        sets["independent test (BanFakeNews-2.0)"] = indep
    results, curves = [], {}

    # %% Baseline: TF-IDF + Logistic Regression | Class-balanced linear model as a reference point.
    # show: pd.concat(results, ignore_index=True) if results else "baseline skipped"
    if cfg.run_baseline:
        pv, *pe = baseline(tr, [va.text] + [d.text for d in sets.values()])
        thr_b = evaluate.best_threshold(va.label, pv)
        bsets = {k: d.assign(p=p) for (k, d), p in zip(sets.items(), pe)}
        results.append(evaluate.table("TF-IDF + LogReg (balanced)", bsets, thr_b))
        curves["TF-IDF + LogReg"] = (te.label, bsets["pooled test"].p)

    # %% Train BanglaBERT (focal loss, resumable) | Each epoch is mirrored to the bucket; a stopped session continues from the last checkpoint. The table shows every epoch, also the ones loaded from the checkpoint (★ = best).
    # show: history(out / "ckpt")
    model, tok = train(cfg, tr, va, out / "ckpt", store)
    # %% Evaluate | Pooled test + independent BanFakeNews-2.0 test. The threshold is chosen on validation.
    # show: res
    # show: by_sub
    thr = evaluate.best_threshold(va.label, predict(model, tok, va.text.tolist(), cfg.max_len))
    msets = {k: d.assign(p=predict(model, tok, d.text.tolist(), cfg.max_len)) for k, d in sets.items()}
    name = f"BanglaBERT ({cfg.loss} loss)"
    results.append(evaluate.table(name, msets, thr))
    curves[name] = (te.label, msets["pooled test"].p)
    res = pd.concat(results, ignore_index=True)
    res.to_csv(out / "tables" / "results.csv", index=False)
    test = msets["pooled test"]
    by_origin = evaluate.by_group(test, "origin", thr)
    by_cat = evaluate.by_group(test, "category", thr)
    by_origin.to_csv(out / "tables" / "by_origin.csv", index=False)
    by_cat.to_csv(out / "tables" / "by_category.csv", index=False)
    by_sub = evaluate.detection_by_subtype(test, thr)
    by_sub.to_csv(out / "tables" / "by_subtype.csv", index=False)
    evaluate.plot_pr_roc(curves, out / "figures" / "roc_pr.png")

    # %% Explain with SHAP | A mix of caught, missed and false-alarm articles: which words push towards "fake".
    # show: g.head(20)
    n = cfg.shap_n
    t = test.assign(pred=(test.p >= thr).astype(int))
    pick = pd.concat([t[(t.label == 1) & (t.pred == 1)].head(n // 2), t[(t.label == 1) & (t.pred == 0)].head(n // 4),
                      t[(t.label == 0) & (t.pred == 1)].head(n // 4)]).head(n)
    sv = explain.explain(model, tok, pick.text.tolist(), cfg)
    (out / "report" / "shap_examples.html").write_text(explain.html_report(sv, pick), encoding="utf-8")
    g = explain.global_tokens(sv)
    g.to_csv(out / "tables" / "shap_tokens.csv", index=False)
    explain.plot_tokens(g, out / "figures" / "shap_tokens.png")

    # %% Export + report | `.pth`, Hugging Face folder, report, optional Hub upload, copy to the bucket.
    md = out / "model"
    model.save_pretrained(md / "hf"); tok.save_pretrained(md / "hf")
    torch.save({"state_dict": model.state_dict(), "base_model": cfg.model_name, "max_len": cfg.max_len,
                "threshold": thr, "id2label": {0: "AUTHENTIC", 1: "FAKE"}}, md / "banglabert_fake_news.pth")
    (md / "hf" / "threshold.json").write_text(json.dumps({"threshold": thr}))
    summary = dict(pool=len(pool), fake_rate=float(pool.label.mean()),
                   counts=pool.groupby(["split", "origin"]).label.agg(["count", "sum"]).reset_index().to_dict("records"),
                   independent_test=len(indep) if indep is not None else 0, threshold=thr,
                   results=res.to_dict("records"), runtime_min=round((time.time() - t0) / 60, 1), config=cfg.to_dict())
    (out / "metrics.json").write_text(json.dumps(summary, indent=1, default=float, ensure_ascii=False))
    from .report import build as build_report
    build_report(out, res, by_origin, by_cat, g, summary, by_sub)

    tok_hf = _hf_token(cfg)
    if tok_hf and (cfg.hub_model_repo or cfg.hub_space_repo):
        from .hub import push_all
        summary["hub"] = push_all(cfg, out, tok_hf)
    if store is not None:
        store.put_dir(out / "report", f"runs/{cfg.run_name}/report")
        store.put(md / "banglabert_fake_news.pth", f"runs/{cfg.run_name}/model/banglabert_fake_news.pth")
    r = res[(res.model == name) & (res.eval_set == "pooled test")].iloc[0]
    log.info("DONE in %.1f min | %s: fake-F1 %.3f macro-F1 %.3f AUROC %.3f", summary["runtime_min"], name,
             r.fake_f1, r.macro_f1, r.auroc)
    return summary


In [ ]:
%%writefile /kaggle/working/repo/src/report.py
from __future__ import annotations

import shutil
from pathlib import Path

import numpy as np
import pandas as pd


def _t(df):
    f = lambda v: "–" if isinstance(v, float) and np.isnan(v) else (f"{v:.3f}" if isinstance(v, float) else str(v))
    return "\n".join(["| " + " | ".join(map(str, df.columns)) + " |", "|" + "---|" * len(df.columns)] +
                     ["| " + " | ".join(f(v) for v in r) + " |" for r in df.itertuples(index=False)])


def build(out: Path, res, by_origin, by_cat, tokens, s, by_sub=None) -> str:
    rep = out / "report"
    (rep / "figures").mkdir(parents=True, exist_ok=True)
    for f in (out / "figures").glob("*.png"):
        shutil.copy(f, rep / "figures" / f.name)
    counts = pd.DataFrame(s["counts"]).rename(columns={"count": "articles", "sum": "fake"})
    cols = ["model", "eval_set", "n", "n_fake", "fake_f1", "macro_f1", "fake_precision", "fake_recall", "auroc", "auprc"]
    md = f"""# Results — Bangla Fake News Detection + Explainability

Data: **BanFakeNews** (Hossain et al., LREC 2020) **+ BanFakeNews-2.0** (Shibu et al., 2025), merged and
**deduplicated across both** ({s['pool']:,} unique articles, {s['fake_rate']:.1%} fake). Articles that also appear in the
BanFakeNews-2.0 independent test set ({s['independent_test']:,}) were removed from training. Split stratified by label × source.
Decision threshold tuned on validation for fake-F1: **{s['threshold']:.2f}**.

## Data
{_t(counts)}

## Results
{_t(res[[c for c in cols if c in res.columns]])}

![ROC / PR](figures/roc_pr.png)

## By source (pooled test) — does the model only learn one dataset's style?
{_t(by_origin) if len(by_origin) else '–'}

## Which kinds of fake news are caught? (pooled test)
BanFakeNews-2.0 marks fake articles with sub-types 0/1/2 (3 = authentic); by inspection type0 looks like satire,
type1 like fabricated news and type2 like clickbait/misleading. `authentic` row = false-alarm rate.

{_t(by_sub) if by_sub is not None and len(by_sub) else '–'}

## By news category (pooled test, ≥50 articles)
{_t(by_cat) if len(by_cat) else '–'}

## Why does it say "fake"? (SHAP)
Global view over the explained test articles (mean SHAP value per word):

![SHAP tokens](figures/shap_tokens.png)

Per-article highlights (red = towards FAKE): **[shap_examples.html](shap_examples.html)**

{_t(tokens.head(15).assign(mean_shap_to_fake=tokens.head(15).mean_shap_to_fake.astype(float)))}

---
Runtime {s['runtime_min']} min · imbalance handled with `{s['config']['loss']}` loss.
"""
    (rep / "REPORT.md").write_text(md, encoding="utf-8")
    return md


In [ ]:
%%writefile /kaggle/working/repo/src/store.py
"""Checkpoint store: a private GCS bucket (auto-created) or a local folder (tests / no GCP).
Everything needed to resume after a Kaggle session stops lives here."""
from __future__ import annotations

import logging
import re
import shutil
from pathlib import Path

log = logging.getLogger(__name__)


def default_bucket_name(project_id: str) -> str:
    name = re.sub(r"[^a-z0-9-]", "-", f"{project_id}-mimic-ckpt".lower()).strip("-")
    return name[:63]


class GCSStore:
    def __init__(self, project_id: str, bucket: str, prefix: str, location: str = "US"):
        from google.cloud import storage

        from .gcp import credentials
        self.client = storage.Client(project=project_id, credentials=credentials("gcs"))
        self.name = (bucket or default_bucket_name(project_id)).replace("gs://", "").strip("/")
        self.prefix = prefix.strip("/")
        self.bucket = self._ensure(project_id, location)

    def _ensure(self, project_id, location):
        from google.api_core import exceptions as gexc
        b = self.client.bucket(self.name)
        try:
            if b.exists():
                return b
        except gexc.Forbidden:
            return b           # exists but owned elsewhere / no get permission; writes will tell
        try:
            b.iam_configuration.uniform_bucket_level_access_enabled = True
            b.iam_configuration.public_access_prevention = "enforced"   # never public
            b = self.client.create_bucket(b, project=project_id, location=location)
            log.info("Created private bucket gs://%s (%s)", self.name, location)
        except gexc.Conflict:
            raise RuntimeError(f"Bucket name '{self.name}' is taken globally. Set cfg.bucket to another name.")
        except Exception as e:
            raise RuntimeError(
                f"Could not create bucket gs://{self.name} ({e}). Create it once in Cloud Shell:\n"
                f"  gcloud storage buckets create gs://{self.name} --project={project_id} --location={location} "
                f"--uniform-bucket-level-access --public-access-prevention\nthen set cfg.bucket='{self.name}'.")
        return b

    def uri(self, key=""):
        return f"gs://{self.name}/{self.prefix}/{key}".rstrip("/")

    def _k(self, key):
        return f"{self.prefix}/{key}"

    def put(self, local, key):
        self.bucket.blob(self._k(key)).upload_from_filename(str(local))

    def get(self, key, local) -> bool:
        bl = self.bucket.blob(self._k(key))
        if not bl.exists():
            return False
        Path(local).parent.mkdir(parents=True, exist_ok=True)
        bl.download_to_filename(str(local))
        return True

    def exists(self, key) -> bool:
        return self.bucket.blob(self._k(key)).exists()

    def list(self, sub) -> list[str]:
        p = self._k(sub).rstrip("/") + "/"
        return [b.name[len(self.prefix) + 1:] for b in self.client.list_blobs(self.bucket, prefix=p)]

    def put_dir(self, local_dir, sub):
        n = 0
        for p in sorted(Path(local_dir).rglob("*")):
            if p.is_file():
                self.put(p, f"{sub.strip('/')}/{p.relative_to(local_dir).as_posix()}")
                n += 1
        log.info("Uploaded %d files -> %s", n, self.uri(sub))


class LocalStore:
    """Same interface backed by a folder (unit tests, or running without GCP)."""

    def __init__(self, root, prefix=""):
        self.root = Path(root) / prefix
        self.root.mkdir(parents=True, exist_ok=True)
        self.name = str(self.root)

    def uri(self, key=""):
        return str(self.root / key)

    def put(self, local, key):
        dst = self.root / key
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(local, dst)

    def get(self, key, local) -> bool:
        src = self.root / key
        if not src.exists():
            return False
        Path(local).parent.mkdir(parents=True, exist_ok=True)
        if Path(local).resolve() != src.resolve():
            shutil.copyfile(src, local)
        return True

    def exists(self, key) -> bool:
        return (self.root / key).exists()

    def list(self, sub) -> list[str]:
        d = self.root / sub
        return [p.relative_to(self.root).as_posix() for p in d.rglob("*") if p.is_file()] if d.exists() else []

    def put_dir(self, local_dir, sub):
        for p in Path(local_dir).rglob("*"):
            if p.is_file():
                self.put(p, f"{sub.strip('/')}/{p.relative_to(local_dir).as_posix()}")


def make_store(cfg):
    if getattr(cfg, "local_store", ""):
        return LocalStore(cfg.local_store, cfg.gcs_prefix)
    from .gcp import validate_project_id
    cfg.project_id = validate_project_id(cfg.project_id)
    s = GCSStore(cfg.project_id, cfg.bucket, cfg.gcs_prefix, cfg.bucket_location)
    cfg.bucket = s.name
    log.info("Checkpoints -> %s", s.uri())
    return s


In [ ]:
%%writefile /kaggle/working/repo/src/train.py
"""BanglaBERT fine-tuning with focal loss (imbalance) and checkpoints mirrored to the GCS bucket (resume)."""
from __future__ import annotations

import logging
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

log = logging.getLogger(__name__)


def focal_loss(logits, y, gamma, alpha_pos):
    ce = F.cross_entropy(logits.float(), y, reduction="none")
    pt = torch.exp(-ce)
    alpha = torch.where(y == 1, torch.full_like(ce, alpha_pos), torch.full_like(ce, 1 - alpha_pos))
    return (alpha * (1 - pt) ** gamma * ce).mean()


def _metrics(ev):
    from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
    logits, y = ev
    p = torch.softmax(torch.tensor(logits).float(), -1)[:, 1].numpy()
    return {"fake_f1": f1_score(y, p >= 0.5), "macro_f1": f1_score(y, p >= 0.5, average="macro"),
            "auroc": roc_auc_score(y, p), "auprc": average_precision_score(y, p)}


def _load(name, **kw):
    from transformers import AutoModelForSequenceClassification
    try:
        m = AutoModelForSequenceClassification.from_pretrained(name, dtype=torch.float32, **kw)
    except TypeError:
        m = AutoModelForSequenceClassification.from_pretrained(name, torch_dtype=torch.float32, **kw)
    return m.float()        # fp32 master weights for mixed precision


def train(cfg, tr: pd.DataFrame, va: pd.DataFrame, out_dir: Path, store=None):
    from datasets import Dataset
    from transformers import (AutoTokenizer, DataCollatorWithPadding, Trainer, TrainerCallback,
                              TrainingArguments)
    tok = AutoTokenizer.from_pretrained(cfg.model_name)
    model = _load(cfg.model_name, num_labels=2, id2label={0: "AUTHENTIC", 1: "FAKE"},
                  label2id={"AUTHENTIC": 0, "FAKE": 1}, ignore_mismatched_sizes=True)
    pos = float(tr.label.mean())
    ce_w = torch.tensor([pos, 1 - pos]) / max(pos, 1 - pos)

    class ImbTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kw):
            y = inputs.pop("labels")
            out = model(**inputs)
            if cfg.loss == "focal":
                loss = focal_loss(out.logits, y, cfg.focal_gamma, cfg.focal_alpha_fake)
            elif cfg.loss == "weighted_ce":
                loss = F.cross_entropy(out.logits.float(), y, weight=ce_w.to(y.device))
            else:
                loss = F.cross_entropy(out.logits.float(), y)
            return (loss, out) if return_outputs else loss

    from .data import fingerprint
    ck = f"ckpt/{cfg.run_name}-{fingerprint(pd.concat([tr, va]))}"     # new data -> fresh checkpoints

    class Mirror(TrainerCallback):
        """Copy each saved checkpoint to the bucket so a dead Kaggle session can resume."""

        def on_save(self, args, state, control, **kw):
            if store is None:
                return
            last = max(Path(args.output_dir).glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[1]))
            store.put_dir(last, f"{ck}/{last.name}")
            (Path(args.output_dir) / "LATEST").write_text(last.name)
            store.put(Path(args.output_dir) / "LATEST", f"{ck}/LATEST")
            log.info("checkpoint %s mirrored to bucket", last.name)

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    if store is not None and not list(out_dir.glob("checkpoint-*")) and store.get(f"{ck}/LATEST", out_dir / "LATEST"):
        name = (out_dir / "LATEST").read_text().strip()
        for k in store.list(f"{ck}/{name}"):
            store.get(k, out_dir / name / Path(k).relative_to(f"{ck}/{name}"))
        log.info("RESUME from bucket checkpoint %s", name)

    def enc(df):
        ds = Dataset.from_pandas(df[["text", "label"]].rename(columns={"label": "labels"}).reset_index(drop=True))
        return ds.map(lambda b: tok(b["text"], truncation=True, max_length=cfg.max_len), batched=True,
                      remove_columns=["text"])
    cuda = torch.cuda.is_available()
    steps = cfg.epochs * -(-len(tr) // (cfg.batch_size * cfg.grad_accum))
    args = TrainingArguments(
        output_dir=str(out_dir), num_train_epochs=cfg.epochs, learning_rate=cfg.lr, weight_decay=0.01,
        per_device_train_batch_size=cfg.batch_size, per_device_eval_batch_size=64,
        gradient_accumulation_steps=cfg.grad_accum, warmup_steps=int(0.06 * steps),
        eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True, metric_for_best_model="fake_f1",
        save_total_limit=1, fp16=cuda, logging_steps=100, report_to=[], seed=cfg.seed, dataloader_num_workers=2)
    trainer = ImbTrainer(model=model, args=args, train_dataset=enc(tr), eval_dataset=enc(va), processing_class=tok,
                         data_collator=DataCollatorWithPadding(tok), compute_metrics=_metrics, callbacks=[Mirror()])
    ckpts = sorted(out_dir.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[1]))
    saved = _saved_epochs(ckpts[-1] if ckpts else None)
    if saved:
        log.info("loaded checkpoint %s: %d of %d epochs already done", ckpts[-1].name, saved, cfg.epochs)
    trainer.train(resume_from_checkpoint=str(ckpts[-1]) if ckpts else None)
    hist = _epoch_table(trainer.state.log_history, "fake_f1", trainer.state.best_metric, saved)
    hist.to_csv(out_dir / "history.csv", index=False)
    log.info("training history (all epochs):\n%s", hist.to_string(index=False) if len(hist) else "-")
    best = trainer.state.best_model_checkpoint
    if best and Path(best).exists():
        # reload with from_pretrained: Trainer's in-place reload misses renamed keys (LayerNorm gamma/beta) on 5.x
        model = _load(best).to(trainer.model.device)
        log.info("best checkpoint reloaded: %s (val fake-F1 %.4f)", Path(best).name, trainer.state.best_metric or float("nan"))
    else:
        model = trainer.model
    return model.eval(), tok


@torch.no_grad()
def predict(model, tok, texts, max_len, bs=64) -> np.ndarray:
    dev = next(model.parameters()).device
    out = []
    for b in range(0, len(texts), bs):
        e = tok(list(texts[b:b + bs]), truncation=True, max_length=max_len, padding=True, return_tensors="pt").to(dev)
        with torch.autocast(dev.type, dtype=torch.float16, enabled=dev.type == "cuda"):
            out.append(torch.softmax(model(**e).logits.float(), -1)[:, 1].cpu().numpy())
    return np.concatenate(out) if out else np.zeros(0)


def baseline(tr: pd.DataFrame, texts_eval: list[pd.Series]):
    """TF-IDF (word 1-2 + char 3-5) + class-balanced logistic regression."""
    import scipy.sparse as sp
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    w = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=200_000, sublinear_tf=True)
    c = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=5, max_features=150_000, sublinear_tf=True)
    t = tr.text.str.slice(0, 2000)
    X = sp.hstack([w.fit_transform(t), c.fit_transform(t)]).tocsr()
    lr = LogisticRegression(C=4.0, class_weight="balanced", max_iter=2000, solver="liblinear").fit(X, tr.label)
    return [lr.predict_proba(sp.hstack([w.transform(s.str.slice(0, 2000)), c.transform(s.str.slice(0, 2000))]).tocsr())[:, 1]
            for s in texts_eval]


def _epoch_table(log_history, best_metric_key, best_metric, saved_epochs):
    """Trainer log_history -> one row per epoch (train loss + validation metrics)."""
    import math
    loss, rows = {}, []
    for h in log_history:
        ep = h.get("epoch")
        if ep is None:
            continue
        if "loss" in h:
            loss[max(1, math.ceil(ep - 1e-9))] = h["loss"]
        if "eval_loss" in h:
            e = int(round(ep))
            r = {"epoch": e, "train_loss": loss.get(e)}
            r.update({k[5:]: v for k, v in h.items() if k.startswith("eval_") and
                      not k.endswith(("runtime", "per_second"))})
            r["source"] = "checkpoint" if e <= saved_epochs else "this session"
            rows.append(r)
    df = pd.DataFrame(rows)
    if len(df) and best_metric is not None and best_metric_key in df:
        hit = np.flatnonzero(np.isclose(df[best_metric_key], best_metric))
        df["best"] = ["★" if i == (hit[0] if len(hit) else -1) else "" for i in range(len(df))]   # first best = kept
    return df


def _saved_epochs(ckpt_dir) -> int:
    import json
    f = Path(ckpt_dir) / "trainer_state.json" if ckpt_dir else None
    return int(round(json.loads(f.read_text()).get("epoch", 0))) if f and f.exists() else 0


def history(out_dir) -> pd.DataFrame:
    """Per-epoch values of the last training in out_dir, including epochs restored from a checkpoint."""
    f = Path(out_dir) / "history.csv"
    return pd.read_csv(f) if f.exists() else pd.DataFrame({"info": ["no training history yet"]})


In [ ]:
%%writefile /kaggle/working/repo/app/__init__.py
"""package"""


In [ ]:
%%writefile /kaggle/working/repo/app/streamlit_app.py
"""Streamlit UI: paste a Bangla news article -> fake probability + SHAP word highlights.

    MODEL_ID=your-hf-user/banglabert-fake-news streamlit run app/streamlit_app.py
"""
import json
import os

import numpy as np
import shap
import streamlit as st
import streamlit.components.v1 as components
import torch
from huggingface_hub import hf_hub_download
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_ID = os.environ.get("MODEL_ID", "model/hf")
st.set_page_config(page_title="বাংলা ফেক নিউজ ডিটেক্টর", page_icon="📰", layout="wide")


@st.cache_resource
def load():
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID).eval()
    try:
        p = os.path.join(MODEL_ID, "threshold.json") if os.path.isdir(MODEL_ID) else hf_hub_download(MODEL_ID, "threshold.json")
        thr = json.load(open(p))["threshold"]
    except Exception:
        thr = 0.5
    return tok, model, thr


tok, model, THR = load()


@torch.no_grad()
def proba(texts):
    e = tok([str(t) for t in texts], truncation=True, max_length=384, padding=True, return_tensors="pt")
    p = torch.softmax(model(**e).logits, -1).numpy()
    return p


st.title("📰 বাংলা ফেক নিউজ ডিটেক্টর")
st.caption("BanglaBERT fine-tuned on BanFakeNews + BanFakeNews-2.0 · SHAP দেখায় কোন শব্দগুলো সিদ্ধান্তে প্রভাব ফেলেছে")
text = st.text_area("শিরোনাম + খবরের লেখা পেস্ট করুন", height=220)
col1, col2 = st.columns([1, 3])
explain = col1.checkbox("SHAP ব্যাখ্যা দেখাও (একটু সময় লাগে)", value=True)
if col1.button("যাচাই করো", type="primary") and text.strip():
    p = float(proba([text])[0, 1])
    verdict = "🚩 সম্ভবত **ভুয়া (FAKE)**" if p >= THR else "✅ সম্ভবত **সঠিক (AUTHENTIC)**"
    col2.metric("P(fake)", f"{p:.1%}", help=f"threshold {THR:.2f}")
    col2.markdown(verdict)
    if explain:
        short = " ".join(text.split()[:150])
        with st.spinner("SHAP হিসাব হচ্ছে..."):
            ex = shap.Explainer(proba, shap.maskers.Text(tok), output_names=["AUTHENTIC", "FAKE"])
            sv = ex([short], max_evals=300)
        st.markdown("লাল = FAKE-এর দিকে ঠেলে, নীল = AUTHENTIC-এর দিকে")
        components.html(shap.plots.text(sv[0, :, 1], display=False), height=420, scrolling=True)
st.info("এটি একটি গবেষণা মডেল — চূড়ান্ত সিদ্ধান্তের আগে নির্ভরযোগ্য সূত্রে যাচাই করুন।")


In [ ]:
%%writefile /kaggle/working/repo/space/Dockerfile
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY streamlit_app.py .
EXPOSE 7860
CMD ["streamlit", "run", "streamlit_app.py", "--server.port=7860", "--server.address=0.0.0.0"]


In [ ]:
%%writefile /kaggle/working/repo/space/MODEL_CARD.md
---
language: [bn]
base_model: csebuetnlp/banglabert
pipeline_tag: text-classification
tags: [fake-news, misinformation, bangla, explainability, shap]
---
# BanglaBERT fake-news classifier

`csebuetnlp/banglabert` fine-tuned on **BanFakeNews + BanFakeNews-2.0**, merged and deduplicated, with focal
loss for class imbalance. Labels: `AUTHENTIC` (0), `FAKE` (1). The decision threshold tuned on validation is in
`threshold.json`.

Evaluation covers a pooled held-out test set and the independent BanFakeNews-2.0 test set; see the GitHub repo's
`results/REPORT.md`, which also includes SHAP explanations.

**Use with care:** the model learns writing style and topic cues from these datasets. It is not a fact-checker.
Check original dataset licences before commercial use.


In [ ]:
%%writefile /kaggle/working/repo/space/README.md
---
title: Bangla Fake News Detector
emoji: 📰
colorFrom: red
colorTo: blue
sdk: docker
app_port: 7860
pinned: false
---
BanglaBERT fake-news classifier with SHAP word-level explanations.


In [ ]:
%%writefile /kaggle/working/repo/space/requirements.txt
--extra-index-url https://download.pytorch.org/whl/cpu
torch
transformers>=4.46
huggingface_hub
shap
streamlit


## 3 · Config
Change only this cell.

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/repo")
from src.config import Config

HF_USER = "YOUR_HF_USERNAME"          # <-- set to publish model + demo (needs HF_TOKEN secret)
publish = HF_USER != "YOUR_HF_USERNAME"
cfg = Config(
    work_dir="/kaggle/working/bfn",
    project_id="mimic-cxr-training",   # GCS bucket for checkpoints/resume ("" to skip)
    loss="focal",                      # focal | weighted_ce | ce
    epochs=3, max_len=384,
    hub_model_repo=f"{HF_USER}/banglabert-fake-news" if publish else "",
    hub_space_repo=f"{HF_USER}/bangla-fake-news-demo" if publish else "",
)

## 4 · Load the pipeline
Imports the project code written above. The cells below are the steps of `src/pipeline.py::run()`, one per cell.

In [ ]:
import sys, time
from IPython.display import display
for _m in [m for m in list(sys.modules) if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]                       # always use the freshly written code (no stale imports)
import src.pipeline as _pipeline
globals().update({k: v for k, v in vars(_pipeline).items() if not k.startswith("__")})   # helpers + imports


class StopRun(Exception):
    """Raised on purpose (e.g. no GPU yet / paused): Run All again later to continue."""


def _stop(result):
    print(result)
    raise StopRun(result.get("status", "stopped") if isinstance(result, dict) else str(result))


roots = None


## 5 · Start
Logging, folders and the checkpoint store.

In [ ]:
t0 = time.time()
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s", force=True)
for noisy in ("httpx", "httpcore", "huggingface_hub", "urllib3", "filelock"):
    logging.getLogger(noisy).setLevel(logging.WARNING)
out = Path(cfg.work_dir)
for d in ("figures", "model", "report", "tables"):
    (out / d).mkdir(parents=True, exist_ok=True)
store = None
if cfg.project_id or cfg.local_store:
    from src.store import make_store
    store = make_store(cfg)

## 6 · Load + merge both datasets
BanFakeNews + BanFakeNews-2.0 → dedupe → stratified split. The independent test set stays separate.

In [ ]:
pool, indep = data.build(cfg, roots)
tr, va, te = (pool[pool.split == s] for s in ("train", "val", "test"))
sets = {"pooled test": te}
if indep is not None and len(indep):
    sets["independent test (BanFakeNews-2.0)"] = indep
results, curves = [], {}
display(pool.groupby(["split", "origin"]).label.agg(["count", "mean"]).rename(columns={"mean": "fake_rate"}))

## 7 · Baseline: TF-IDF + Logistic Regression
Class-balanced linear model as a reference point.

In [ ]:
if cfg.run_baseline:
    pv, *pe = baseline(tr, [va.text] + [d.text for d in sets.values()])
    thr_b = evaluate.best_threshold(va.label, pv)
    bsets = {k: d.assign(p=p) for (k, d), p in zip(sets.items(), pe)}
    results.append(evaluate.table("TF-IDF + LogReg (balanced)", bsets, thr_b))
    curves["TF-IDF + LogReg"] = (te.label, bsets["pooled test"].p)
display(pd.concat(results, ignore_index=True) if results else "baseline skipped")

## 8 · Train BanglaBERT (focal loss, resumable)
Each epoch is mirrored to the bucket; a stopped session continues from the last checkpoint. The table shows every epoch, also the ones loaded from the checkpoint (★ = best).

In [ ]:
model, tok = train(cfg, tr, va, out / "ckpt", store)
display(history(out / "ckpt"))

## 9 · Evaluate
Pooled test + independent BanFakeNews-2.0 test. The threshold is chosen on validation.

In [ ]:
thr = evaluate.best_threshold(va.label, predict(model, tok, va.text.tolist(), cfg.max_len))
msets = {k: d.assign(p=predict(model, tok, d.text.tolist(), cfg.max_len)) for k, d in sets.items()}
name = f"BanglaBERT ({cfg.loss} loss)"
results.append(evaluate.table(name, msets, thr))
curves[name] = (te.label, msets["pooled test"].p)
res = pd.concat(results, ignore_index=True)
res.to_csv(out / "tables" / "results.csv", index=False)
test = msets["pooled test"]
by_origin = evaluate.by_group(test, "origin", thr)
by_cat = evaluate.by_group(test, "category", thr)
by_origin.to_csv(out / "tables" / "by_origin.csv", index=False)
by_cat.to_csv(out / "tables" / "by_category.csv", index=False)
by_sub = evaluate.detection_by_subtype(test, thr)
by_sub.to_csv(out / "tables" / "by_subtype.csv", index=False)
evaluate.plot_pr_roc(curves, out / "figures" / "roc_pr.png")
display(res)
display(by_sub)

## 10 · Explain with SHAP
A mix of caught, missed and false-alarm articles: which words push towards "fake".

In [ ]:
n = cfg.shap_n
t = test.assign(pred=(test.p >= thr).astype(int))
pick = pd.concat([t[(t.label == 1) & (t.pred == 1)].head(n // 2), t[(t.label == 1) & (t.pred == 0)].head(n // 4),
                  t[(t.label == 0) & (t.pred == 1)].head(n // 4)]).head(n)
sv = explain.explain(model, tok, pick.text.tolist(), cfg)
(out / "report" / "shap_examples.html").write_text(explain.html_report(sv, pick), encoding="utf-8")
g = explain.global_tokens(sv)
g.to_csv(out / "tables" / "shap_tokens.csv", index=False)
explain.plot_tokens(g, out / "figures" / "shap_tokens.png")
display(g.head(20))

## 11 · Export + report
`.pth`, Hugging Face folder, report, optional Hub upload, copy to the bucket.

In [ ]:
md = out / "model"
model.save_pretrained(md / "hf"); tok.save_pretrained(md / "hf")
torch.save({"state_dict": model.state_dict(), "base_model": cfg.model_name, "max_len": cfg.max_len,
            "threshold": thr, "id2label": {0: "AUTHENTIC", 1: "FAKE"}}, md / "banglabert_fake_news.pth")
(md / "hf" / "threshold.json").write_text(json.dumps({"threshold": thr}))
summary = dict(pool=len(pool), fake_rate=float(pool.label.mean()),
               counts=pool.groupby(["split", "origin"]).label.agg(["count", "sum"]).reset_index().to_dict("records"),
               independent_test=len(indep) if indep is not None else 0, threshold=thr,
               results=res.to_dict("records"), runtime_min=round((time.time() - t0) / 60, 1), config=cfg.to_dict())
(out / "metrics.json").write_text(json.dumps(summary, indent=1, default=float, ensure_ascii=False))
from src.report import build as build_report
build_report(out, res, by_origin, by_cat, g, summary, by_sub)

tok_hf = _hf_token(cfg)
if tok_hf and (cfg.hub_model_repo or cfg.hub_space_repo):
    from src.hub import push_all
    summary["hub"] = push_all(cfg, out, tok_hf)
if store is not None:
    store.put_dir(out / "report", f"runs/{cfg.run_name}/report")
    store.put(md / "banglabert_fake_news.pth", f"runs/{cfg.run_name}/model/banglabert_fake_news.pth")
r = res[(res.model == name) & (res.eval_set == "pooled test")].iloc[0]
log.info("DONE in %.1f min | %s: fake-F1 %.3f macro-F1 %.3f AUROC %.3f", summary["runtime_min"], name,
         r.fake_f1, r.macro_f1, r.auroc)
result = summary
result

## 12 · Results report
Aggregate results only (safe for GitHub).

In [ ]:
import re, shutil, glob
from IPython.display import Markdown, Image, HTML, display
rep = f"{cfg.work_dir}/report"
md = open(f"{rep}/REPORT.md", encoding="utf-8").read()
display(Markdown(re.sub(r"!\[.*?\]\(figures/.*?\)", "", md)))
for f in sorted(glob.glob(f"{rep}/figures/*.png")):
    display(Image(f))
display(HTML(open(f"{rep}/shap_examples.html", encoding="utf-8").read()[:200000]))
shutil.make_archive("/kaggle/working/results_for_github", "zip", rep)
print("Results zip  : /kaggle/working/results_for_github.zip -> unzip into the repo as results/")
print("Model (.pth) :", f"{cfg.work_dir}/model/banglabert_fake_news.pth")